# Session 5 — Text Embeddings and Qdrant Similarity Search
Objective

In this notebook, we will:

Load the cleaned dataset from previous ETL notebooks
Select a single text field for semantic embedding
Generate sentence embeddings using Sentence-BERT
Store vectors inside Qdrant
Perform semantic similarity search
Save embedding artifacts for Streamlit integration

In [1]:
# ============================================================
# Imports
# ============================================================

import pandas as pd
import numpy as np
import os

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

In [2]:
# ============================================================
# Project Paths
# ============================================================

BASE_DIR = ".."

CLEAN_OUTPUT_DIR = f"{BASE_DIR}/data/clean"

ARTIFACT_DIR = f"{BASE_DIR}/artifacts"

os.makedirs(ARTIFACT_DIR, exist_ok=True)

print("Artifacts directory ready.")

Artifacts directory ready.


In [3]:


main_df = pd.read_parquet(f"{CLEAN_OUTPUT_DIR}/merged_df.parquet")

print("Dataset Shape:", main_df.shape)

main_df.head()

Dataset Shape: (114187, 55)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode,delay,late_flag
0,DEBIT,3,4,91.25,314.6400146,Advance shipping,0,73,Sporting Goods,Caguas,...,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class,-1,0
1,DEBIT,3,4,22.86000061,304.8099976,Advance shipping,0,73,Sporting Goods,Los Angeles,...,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class,-1,0
2,PAYMENT,2,4,134.2100067,298.25,Advance shipping,0,73,Sporting Goods,Caguas,...,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class,-2,0
3,TRANSFER,6,4,18.57999992,294.980011,Shipping canceled,0,73,Sporting Goods,Tonawanda,...,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/19/2018 11:03,Standard Class,2,1
4,DEBIT,2,1,95.18000031,288.4200134,Late delivery,1,73,Sporting Goods,Caguas,...,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 10:42,First Class,1,1


In [4]:
# ============================================================
# 3. CREATE NODE ID
# ============================================================

main_df = main_df.reset_index(drop=True)

main_df["node_id"] = main_df["Order Id"].astype(str)

main_df[["node_id"]].head()

,node_id
0,77202
1,75937
2,75936
3,75935
4,75934


## Select Semantic Text Columns

We only select columns that contain meaningful semantic information.

Chosen columns:
- `Category Name`
- `Department Name`
- `Product Name`

In [5]:


text_columns = [
    "Category Name",
    "Department Name",
    "Product Name"
]

main_df[text_columns] = (
    main_df[text_columns]
    .fillna("")
    .astype(str)
    .apply(lambda col: col.str.lower().str.strip())
)

main_df["combined_text"] = (
    main_df["Category Name"] + " | " +
    main_df["Department Name"] + " | " +
    main_df["Product Name"] 
)

main_df[["node_id", "combined_text"]].head()

,node_id,combined_text
0,77202,sporting goods | fitness | smart watch
1,75937,sporting goods | fitness | smart watch
2,75936,sporting goods | fitness | smart watch
3,75935,sporting goods | fitness | smart watch
4,75934,sporting goods | fitness | smart watch


## Load Sentence Transformer Model

We use:

`all-MiniLM-L6-v2`

Characteristics:
- 384-dimensional embeddings
- optimized for semantic similarity
- lightweight and fast
- widely used for retrieval systems

In [6]:
# ============================================================
# Load Model
# ============================================================

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Model Loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model Loaded


## Generate Text Embeddings

Each text row is converted into a dense numerical vector.

Rows with similar meanings will produce similar vectors.

In [7]:
texts = main_df["combined_text"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

print("Embeddings Shape:", embeddings.shape)

Batches:   0%|          | 0/3569 [00:00<?, ?it/s]

Embeddings Shape: (114187, 384)


In [8]:
os.makedirs(CLEAN_OUTPUT_DIR, exist_ok=True)

np.save(
    f"{CLEAN_OUTPUT_DIR}/product_embeddings.npy",
    embeddings
)

print("Embeddings saved successfully.")

Embeddings saved successfully.



# Connect to Qdrant

We now connect to the local Qdrant instance running inside Docker.

In [9]:
client = QdrantClient(
    host="localhost",
    port=6333
)

print("Connected to Qdrant successfully.")

Connected to Qdrant successfully.


In [10]:

# ============================================================
# 11. CREATE QDRANT COLLECTION
# ============================================================

collection_name = "supply_chain_products"
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

print("Old collection removed.")
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

print("Qdrant collection created successfully.")

Old collection removed.
Qdrant collection created successfully.


## Prepare Points for Qdrant

Each vector point contains:
- unique ID
- embedding vector
- payload metadata

Payload metadata helps:
- display results
- filter results
- connect retrieval results back to original records

In [11]:


points = []

for i in range(len(main_df)):

    point = PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={
            "node_id": main_df.iloc[i]["node_id"],
            "raw_text": main_df.iloc[i]["combined_text"]
        }
    )

    points.append(point)

print("Total points prepared:", len(points))

Total points prepared: 114187


# Batch Upsert Embeddings into Qdrant

Uploading all embeddings in one request can exceed Qdrant's payload size limit.

To avoid this, we upload embeddings in smaller batches.

This is also the standard production approach for large vector datasets.


In [12]:
BATCH_SIZE = 1000

for i in range(0, len(points), BATCH_SIZE):

    batch = points[i:i+BATCH_SIZE]

    client.upsert(
        collection_name=collection_name,
        points=batch
    )

    print(f"Uploaded batch {i // BATCH_SIZE + 1}")

Uploaded batch 1
Uploaded batch 2
Uploaded batch 3
Uploaded batch 4
Uploaded batch 5
Uploaded batch 6
Uploaded batch 7
Uploaded batch 8
Uploaded batch 9
Uploaded batch 10
Uploaded batch 11
Uploaded batch 12
Uploaded batch 13
Uploaded batch 14
Uploaded batch 15
Uploaded batch 16
Uploaded batch 17
Uploaded batch 18
Uploaded batch 19
Uploaded batch 20
Uploaded batch 21
Uploaded batch 22
Uploaded batch 23
Uploaded batch 24
Uploaded batch 25
Uploaded batch 26
Uploaded batch 27
Uploaded batch 28
Uploaded batch 29
Uploaded batch 30
Uploaded batch 31
Uploaded batch 32
Uploaded batch 33
Uploaded batch 34
Uploaded batch 35
Uploaded batch 36
Uploaded batch 37
Uploaded batch 38
Uploaded batch 39
Uploaded batch 40
Uploaded batch 41
Uploaded batch 42
Uploaded batch 43
Uploaded batch 44
Uploaded batch 45
Uploaded batch 46
Uploaded batch 47
Uploaded batch 48
Uploaded batch 49
Uploaded batch 50
Uploaded batch 51
Uploaded batch 52
Uploaded batch 53
Uploaded batch 54
Uploaded batch 55
Uploaded batch 56
U

# Step 9 — Run a Similarity Search Query

Now we test the retrieval pipeline.

The process works as follows:

1. User enters a query text
2. The same embedding model converts the query into a vector
3. Qdrant compares the query vector against stored vectors
4. The most semantically similar records are returned

This forms the Retrieval component of the RAG-style architecture.

In [13]:
query_text = "running shoes for fitness training"

query_vector = model.encode(query_text).tolist()

print("Query embedded successfully.")

Query embedded successfully.


# Retrieve Top Similar Results

We now search the Qdrant collection using cosine similarity.

The top matching vectors represent records that are semantically similar to the query text.

In [14]:
results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=5,
    with_payload=True
)

# Display Similar Results

Each result contains:
- similarity score
- node_id
- original raw text

Higher cosine similarity scores indicate stronger semantic similarity.

In [15]:

# ============================================================
# 18. DISPLAY RESULTS
# ============================================================

for idx, hit in enumerate(results.points):

    print(f"\nResult {idx + 1}")
    print("-" * 60)

    print("Similarity Score:", round(hit.score, 4))

    print("Node ID:")
    print(hit.payload["node_id"])

    print("\nSimilar Text:")
    print(hit.payload["raw_text"])


Result 1
------------------------------------------------------------
Similarity Score: 0.6271
Node ID:
51573

Similar Text:
cardio equipment | footwear | nike men's free 5.0+ running shoe

Result 2
------------------------------------------------------------
Similarity Score: 0.6271
Node ID:
1194

Similar Text:
cardio equipment | footwear | nike men's free 5.0+ running shoe

Result 3
------------------------------------------------------------
Similarity Score: 0.6271
Node ID:
1397

Similar Text:
cardio equipment | footwear | nike men's free 5.0+ running shoe

Result 4
------------------------------------------------------------
Similarity Score: 0.6271
Node ID:
52741

Similar Text:
cardio equipment | footwear | nike men's free 5.0+ running shoe

Result 5
------------------------------------------------------------
Similarity Score: 0.6271
Node ID:
60601

Similar Text:
cardio equipment | footwear | nike men's free 5.0+ running shoe


In [16]:
# Check what is actually stored in Qdrant payload
results = client.scroll(
    collection_name="supply_chain_products",
    limit=3,
    with_payload=True,
    with_vectors=False
)

for point in results[0]:
    print("ID:", point.id)
    print("Payload:", point.payload)
    print()

ID: 0
Payload: {'node_id': '77202', 'raw_text': 'sporting goods | fitness | smart watch'}

ID: 1
Payload: {'node_id': '75937', 'raw_text': 'sporting goods | fitness | smart watch'}

ID: 2
Payload: {'node_id': '75936', 'raw_text': 'sporting goods | fitness | smart watch'}

